# Field Scan : affinage de DeepForest sur les bananiers (ReforesTree)

**Entrée** : `deepforest.zip` (tuiles de 400 × 400 px à 10 cm/pixel et boîtes « Banana »), produit par `scripts/preparer_deepforest.py`, déposé dans `Mon Drive/field_scan/`.

**Sortie** (dans `Mon Drive/field_scan/modeles/`) : les poids du modèle affiné (`.ckpt` et `.pt`) et un fichier `entrainement.json` (versions, réglages, scores).

**Découpage** : entraînement sur Flora Pluas, Nestor Macias, Leonor Aspiazu ; suivi sur Manuel Macias (annotations ReforesTree, incomplètes). La vraie mesure se fait ensuite sur ton ordinateur, contre la référence manuelle (MM1, MM2), avec `evaluer_comptage.py`.

Avant de commencer : **Exécution > Modifier le type d'exécution > GPU (T4)**.

## 1. Google Drive et données

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil
DOSSIER_DRIVE = '/content/drive/MyDrive/field_scan'
ZIP = f'{DOSSIER_DRIVE}/deepforest.zip'
DONNEES = '/content/deepforest'
MODELES = f'{DOSSIER_DRIVE}/modeles'
os.makedirs(MODELES, exist_ok=True)

assert os.path.exists(ZIP), f"Introuvable : {ZIP}. Dépose deepforest.zip dans Mon Drive/field_scan/"
shutil.rmtree(DONNEES, ignore_errors=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(DONNEES)
print(sorted(os.listdir(DONNEES)))
print(len(os.listdir(f'{DONNEES}/images')), 'tuiles')

## 2. Installation et versions

On installe la version courante de DeepForest et on **note sa version** : elle ira dans le carnet de données, pour la reproductibilité.

In [ ]:
!pip -q install deepforest
import torch, deepforest, json, platform
from importlib.metadata import version
VERSIONS = {'deepforest': version('deepforest'), 'torch': torch.__version__, 'python': platform.python_version(),
            'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'aucun'}
print(VERSIONS)
assert torch.cuda.is_available(), "Pas de GPU : Exécution > Modifier le type d'exécution > GPU" 

## 3. Annotations

Les bananiers sont renommés « Tree », la classe du modèle pré-entraîné : on garde sa tête de détection telle quelle. Une ligne à coordonnées nulles est une tuile sans bananier (exemple négatif).

In [ ]:
import pandas as pd
train = pd.read_csv(f'{DONNEES}/train.csv'); val = pd.read_csv(f'{DONNEES}/val.csv')
for df in (train, val):
    df['label'] = 'Tree'
train.to_csv(f'{DONNEES}/train_tree.csv', index=False); val.to_csv(f'{DONNEES}/val_tree.csv', index=False)
for nom, df in (('entraînement', train), ('suivi', val)):
    vides = ((df.xmax == 0) & (df.ymax == 0)).sum()
    print(f"{nom} : {df.image_path.nunique()} tuiles, {len(df) - vides} boîtes, {vides} tuiles sans bananier, "
          f"côté médian {((df.xmax - df.xmin)[df.xmax > 0]).median():.0f} px")

In [ ]:
# Contrôle visuel : une tuile d'entraînement et ses boîtes
import matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image
nom = train.image_path.value_counts().index[0]
fig, ax = plt.subplots(figsize=(7, 7)); ax.imshow(Image.open(f'{DONNEES}/images/{nom}'))
for _, r in train[train.image_path == nom].iterrows():
    ax.add_patch(patches.Rectangle((r.xmin, r.ymin), r.xmax - r.xmin, r.ymax - r.ymin, fill=False, color='yellow'))
ax.set_title(nom); ax.axis('off'); plt.show()

## 4. Modèle pré-entraîné et score de départ

On mesure d'abord le modèle **sans affinage** sur les tuiles de suivi : c'est le point de comparaison de l'affinage.

In [ ]:
from deepforest import main
modele = main.deepforest()
try:
    modele.load_model(model_name='weecology/deepforest-tree', revision='main')
except (AttributeError, TypeError):
    modele.use_release()   # anciennes versions de DeepForest
print('Classes :', getattr(modele, 'label_dict', None))

def evaluer_tuiles(m, csv, iou=0.4):
    '''Précision et rappel des boîtes sur des tuiles (référence ReforesTree, incomplète : simple suivi).'''
    try:
        res = m.evaluate(csv_file=csv, root_dir=f'{DONNEES}/images', iou_threshold=iou)
    except TypeError:
        res = m.evaluate(csv, f'{DONNEES}/images', iou_threshold=iou)
    return {k: float(res[k]) for k in ('box_precision', 'box_recall') if k in res}

SCORE_DEPART = evaluer_tuiles(modele, f'{DONNEES}/val_tree.csv')
print('Sans affinage :', SCORE_DEPART)

## 5. Affinage

Réglages modestes, adaptés à 56 tuiles. Le meilleur modèle est choisi sur les tuiles de suivi, jamais sur le test.

In [ ]:
REGLAGES = dict(epochs=30, batch_size=4, lr=0.0005, score_thresh=0.3)

c = modele.config
c['train']['csv_file'] = f'{DONNEES}/train_tree.csv'
c['train']['root_dir'] = f'{DONNEES}/images'
c['validation']['csv_file'] = f'{DONNEES}/val_tree.csv'
c['validation']['root_dir'] = f'{DONNEES}/images'
c['train']['epochs'] = REGLAGES['epochs']
c['train']['lr'] = REGLAGES['lr']
c['batch_size'] = REGLAGES['batch_size']
c['score_thresh'] = REGLAGES['score_thresh']
c['workers'] = 2
try:
    c['validation']['val_accuracy_interval'] = 5   # précision et rappel de suivi toutes les 5 époques
except Exception:
    pass

from pytorch_lightning.callbacks import ModelCheckpoint
point_controle = ModelCheckpoint(dirpath=MODELES, filename='bananiers-{epoch:02d}-{val_classification:.3f}',
                                 monitor='val_classification', mode='min', save_top_k=1, save_last=True)
try:
    modele.create_trainer(callbacks=[point_controle], log_every_n_steps=5)
except TypeError:
    modele.create_trainer()
modele.trainer.fit(modele)
print('Meilleur point de contrôle :', point_controle.best_model_path)

## 6. Score après affinage et enregistrement

In [ ]:
SCORE_FINAL = evaluer_tuiles(modele, f'{DONNEES}/val_tree.csv')
print('Sans affinage :', SCORE_DEPART); print('Après affinage :', SCORE_FINAL)

chemin_ckpt = f'{MODELES}/bananiers_deepforest.ckpt'
modele.trainer.save_checkpoint(chemin_ckpt)
torch.save(modele.model.state_dict(), f'{MODELES}/bananiers_deepforest_poids.pt')
json.dump(dict(versions=VERSIONS, reglages=REGLAGES, resolution_cm=10, taille_tuile_px=400,
               donnees=json.load(open(f'{DONNEES}/parametres.json')),
               score_suivi_sans_affinage=SCORE_DEPART, score_suivi_apres_affinage=SCORE_FINAL,
               meilleur_point_controle=point_controle.best_model_path,
               remarque='Suivi sur annotations ReforesTree de Manuel Macias (incomplètes) ; '
                        'la mesure de référence se fait contre la référence manuelle MM1 et MM2.'),
          open(f'{MODELES}/entrainement.json', 'w'), indent=2, ensure_ascii=False)
print('Enregistré dans', MODELES, ':', sorted(os.listdir(MODELES)))

## 7. Contrôle visuel sur une tuile de suivi

In [ ]:
nom = val.image_path.value_counts().index[0]
pred = modele.predict_image(path=f'{DONNEES}/images/{nom}')
fig, ax = plt.subplots(figsize=(7, 7)); ax.imshow(Image.open(f'{DONNEES}/images/{nom}'))
for _, r in val[(val.image_path == nom) & (val.xmax > 0)].iterrows():
    ax.add_patch(patches.Rectangle((r.xmin, r.ymin), r.xmax - r.xmin, r.ymax - r.ymin, fill=False, color='yellow'))
if pred is not None:
    for _, r in pred.iterrows():
        ax.add_patch(patches.Rectangle((r.xmin, r.ymin), r.xmax - r.xmin, r.ymax - r.ymin, fill=False, color='red'))
ax.set_title(f'{nom} : jaune = ReforesTree, rouge = modèle'); ax.axis('off'); plt.show()

## 8. Ensuite, sur ton ordinateur

Télécharge depuis `Mon Drive/field_scan/modeles/` les fichiers `bananiers_deepforest.ckpt` et `entrainement.json`, et range-les dans `Field-Scan/modeles/`. L'évaluation sur la référence manuelle se fera sur processeur, comme l'exige la consigne pour l'inférence.